# MeshVTON — Çıkarım (inference)

Eğitilmiş Aşama-1 checkpoint'iyle: kullanıcının yüklediği KENDİ fotoğrafı + `data/garments_3d/upper_body`
kütüphanesinden seçilen bir 3D giysi → try-on sonucu.

**Not:** Model yalnızca 3D mesh + gri (texture'sız) koşullama ile eğitildi (bkz. eğitim notebook'u,
KALICI no-texture kuralı). Giysi seçimi bu yüzden bir FOTOĞRAF değil, mevcut mesh kütüphanesinden bir
klasör adı (`GARMENT_ID`) — kullanıcı yalnızca KENDİ fotoğrafını sağlıyor.

In [ ]:
#@title 1) Kurulum (train notebook'uyla aynı — pyrender/HMR2/IDM-VTON preprocess)
import os
if not os.path.exists('/content/MeshVTON'):
    !git clone https://github.com/SerhanTelatar/MeshVTON /content/MeshVTON
%cd /content/MeshVTON
!git pull

!pip -q install "diffusers>=0.34" "peft>=0.14" lpips einops sentencepiece trimesh smplx pyrender onnxruntime
!pip -q uninstall -y pyopengl PyOpenGL-accelerate > /dev/null 2>&1
!pip -q install "git+https://github.com/mmatl/pyopengl.git"
import importlib.util
if importlib.util.find_spec('hmr2') is None:
    !pip -q install "git+https://github.com/shubham-goel/4D-Humans.git"
!apt-get -qq install -y libglu1-mesa libosmesa6 > /dev/null 2>&1
import subprocess
_probe = subprocess.run(["python", "-c",
    'import os;os.environ["PYOPENGL_PLATFORM"]="egl";'
    'import pyrender;r=pyrender.OffscreenRenderer(16,16);r.delete();print("egl-ok")'],
    capture_output=True, text=True)
os.environ['PYOPENGL_PLATFORM'] = 'egl' if 'egl-ok' in _probe.stdout else 'osmesa'
print('GL platform:', os.environ['PYOPENGL_PLATFORM'])

if not os.path.exists('/content/IDM-VTON'):
    !git clone -q https://github.com/yisol/IDM-VTON /content/IDM-VTON
import shutil
from huggingface_hub import hf_hub_download
for repo_path in ('humanparsing/parsing_atr.onnx',
                  'humanparsing/parsing_lip.onnx',
                  'openpose/ckpts/body_pose_model.pth'):
    local = f'/content/IDM-VTON/ckpt/{repo_path}'
    if not (os.path.exists(local) and os.path.getsize(local) > 1_000_000):
        os.makedirs(os.path.dirname(local), exist_ok=True)
        shutil.copy(hf_hub_download('yisol/IDM-VTON', repo_path), local)
    assert os.path.getsize(local) > 1_000_000, f'bozuk indirme: {local}'

from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('Kurulum OK')

In [ ]:
#@title 2) Veri + checkpoint — Drive/MeshVTON düzeni
import os, sys
sys.path.insert(0, '/content/MeshVTON/v2')
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/MeshVTON'

# Giysi kütüphanesi (yalnız upper_body kullanılıyor — model bunlarla eğitildi)
!mkdir -p /content/MeshVTON/data
!unzip -q -n $D/garments_3d.zip -d /content/MeshVTON/data   # -> data/garments_3d/

# SMPL-X gövde modeli
!mkdir -p /content/MeshVTON/checkpoints/pretrained/smplx
!cp $D/smplx/SMPLX_NEUTRAL.* /content/MeshVTON/checkpoints/pretrained/smplx/
os.environ['SMPLX_MODEL_DIR'] = '/content/MeshVTON/checkpoints/pretrained/smplx'

# HMR2 ağırlıkları + SMPL neutral pkl
import glob, shutil
from meshvton2.conditioning.body import _patch_torch_load_weights_only
import torch as _torch; _patch_torch_load_weights_only(_torch)
from hmr2.models import download_models
from hmr2.configs import CACHE_DIR_4DHUMANS
download_models(CACHE_DIR_4DHUMANS)
smpl_dir = f"{CACHE_DIR_4DHUMANS}/data/smpl"; os.makedirs(smpl_dir, exist_ok=True)
cands = glob.glob(f'{D}/smpl/*neutral*lbs*.pkl') + glob.glob(f'{D}/smpl/SMPL_NEUTRAL.pkl')
assert cands, "SMPL neutral pkl yok -> Drive/MeshVTON/smpl/"
shutil.copy(cands[0], f"{smpl_dir}/SMPL_NEUTRAL.pkl")

# Eğitilmiş checkpoint — istenirse değiştirin
CHECKPOINT = f"{D}/v2_outputs/stage1/final.pt"  #@param {type:"string"}
assert os.path.exists(CHECKPOINT), f"checkpoint bulunamadı: {CHECKPOINT}"
print('Veri + checkpoint hazır:', CHECKPOINT)

In [ ]:
#@title 3) Kişi fotoğraflarını yükleyin (BİRDEN ÇOK seçebilirsiniz)
from google.colab import files
from PIL import Image
import pathlib

uploaded = files.upload()  # Ctrl/Cmd ile birden çok dosya seçin

# Baytları KENDİMİZ yazıyoruz. files.upload() dosyayı ÇALIŞMA DİZİNİNE koyuyor —
# hücre 1'deki `%cd /content/MeshVTON` yüzünden /content DEĞİL — ve aynı ad varsa
# "ad (2).png" diye yeniden adlandırıyor. Yol tahmin etmek bu yüzden kırılgandı.
UPLOAD_DIR = pathlib.Path('/content/uploads')
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
PERSON_IMAGES = []
for name, data in uploaded.items():
    p = UPLOAD_DIR / pathlib.Path(name).name
    p.write_bytes(data)
    PERSON_IMAGES.append(str(p))
assert PERSON_IMAGES, "hiç dosya yüklenmedi"

# ÖNEMLİ: ön-işleme fotoğrafı KOŞULSUZ 768x1024'e resize eder (person.py:119) —
# en-boy oranı KORUNMAZ, kırpma da yok. 3:4 dışındaki foto GERİLİR; gerilmiş gövdede
# HMR2 pozu ve parser maskesi bozulur. Kişi içermeyen görseller hücre 5'te atlanır.
print(f"\n{len(PERSON_IMAGES)} fotoğraf hazır:\n")
for p in list(PERSON_IMAGES):
    try:
        w, h = Image.open(p).size
    except Exception as e:
        print(f"  {pathlib.Path(p).name:45s} AÇILAMADI ({e}) — listeden çıkarıldı")
        PERSON_IMAGES.remove(p)
        continue
    note = "OK (3:4)" if abs(w / h - 0.75) < 0.03 else \
        f"DİKKAT: oran {w / h:.2f}, 3:4=0.75 → GERİLECEK, önce kırpın"
    if w < 768 or h < 1024:
        note += "  |  düşük çözünürlük (<768x1024)"
    print(f"  {pathlib.Path(p).name:45s} {w}x{h}  {note}")

In [ ]:
#@title 4) Giysi(ler) seçin — virgülle birden çok yazabilirsiniz
import pathlib
GARMENTS_ROOT = pathlib.Path('/content/MeshVTON/data/garments_3d')
available = sorted(p.parent for p in (GARMENTS_ROOT / 'upper_body').rglob('*.obj'))
print(f'{len(available)} giysi bulundu, ilk 10:')
for d in available[:10]:
    print(' ', d.relative_to(GARMENTS_ROOT))

GARMENT_IDS = "upper_body/00047_Top, upper_body/00111_Tshirt"  #@param {type:"string"}
GARMENT_LIST = [g.strip() for g in GARMENT_IDS.split(',') if g.strip()]
missing = [g for g in GARMENT_LIST if not (GARMENTS_ROOT / g).exists()]
assert not missing, f'giysi klasörü yok: {missing}'
print(f'\n{len(GARMENT_LIST)} giysi seçildi: {GARMENT_LIST}')
print(f'toplam üretilecek: {len(PERSON_IMAGES)} foto × {len(GARMENT_LIST)} giysi = '
      f'{len(PERSON_IMAGES) * len(GARMENT_LIST)} görsel (~{len(PERSON_IMAGES) * len(GARMENT_LIST) * 19} sn)')

In [ ]:
#@title 5) Üret — her fotoğraf × her giysi (model BİR kez yüklenir)
STEPS    = 28   #@param {type:"integer"}
GUIDANCE = 1.0  #@param {type:"number"}
SEED     = 0    #@param {type:"integer"}
# GUIDANCE: 1.0 = eğitimdeki değer (varsayılan). FLUX Fill guidance-damıtılmış;
# 3.5-30 aralığı sadakat/doygunluğu artırır — çıktı SOLGUN geliyorsa ilk denenecek kol.
# SEED sabit tutulursa giysiler arası tek değişken mesh olur (tez figürü için şart).

import sys, yaml, pathlib
sys.path.insert(0, '/content/MeshVTON/v2')
from PIL import Image
from IPython.display import display, Markdown
from meshvton2.conditioning.body import build_hmr2_backend
from meshvton2.conditioning.builder import PhotoView, assert_real_impl, build_conditioning
from meshvton2.conditioning.garment import load_garment_asset
from meshvton2.conditioning.person import PersonPreprocessor, person_square_bbox
from meshvton2.model.flux_tryon import FluxTryOnSampler

assert_real_impl()
base = yaml.safe_load(open('/content/MeshVTON/v2/configs/base.yaml'))
size = (base['resolution']['height'], base['resolution']['width'])
OUT = pathlib.Path('/content/MeshVTON/v2/outputs/inference'); OUT.mkdir(parents=True, exist_ok=True)

# Pahalı nesneler BİR kez kurulur — FLUX yüklemesi her foto için tekrarlanmasın
prep = PersonPreprocessor('/content/IDM-VTON')
hmr2 = build_hmr2_backend()
sampler = FluxTryOnSampler(base['model']['flux_fill_repo'], checkpoint=CHECKPOINT,
                           prompt=base['model']['prompt'])

# HAM asset — builder force_textureless'i kendi uyguluyor (builder.py:316,341)
assets = {gid: load_garment_asset(sorted((GARMENTS_ROOT / gid).glob('*.obj'))[0],
                                  garment_id=gid.replace('/', '__'), allow_untextured=True)
          for gid in GARMENT_LIST}

results, persons = {}, {}
total, i = len(PERSON_IMAGES) * len(GARMENT_LIST), 0
for img_path in PERSON_IMAGES:
    name = pathlib.Path(img_path).stem
    try:
        pp = prep.process(img_path, size=size)
        params = hmr2(pp.image, bbox=person_square_bbox(pp))  # kişi-merkezli kare bbox (hizalama)
    except Exception as e:
        # tek fotoğrafın hatası tüm koşuyu öldürmesin (boş maske = kişi tespit edilemedi)
        print(f'ATLA {name}: kişi ön-işleme başarısız — {e}')
        i += len(GARMENT_LIST)
        continue
    persons[name] = pp
    for gid in GARMENT_LIST:
        i += 1
        try:
            bundle = build_conditioning(pp.image, params, assets[gid], PhotoView(),
                                        size=size, person_prep=pp)
            out = sampler.sample(bundle, steps=STEPS, seed=SEED,
                                 control_scale=1.0, guidance=GUIDANCE)
            results[(name, gid)] = out
            fn = OUT / f"{name}__{gid.replace('/', '__')}.png"
            Image.fromarray(out).save(fn)
            print(f'[{i}/{total}] OK {name} × {gid}')
        except Exception as e:
            print(f'[{i}/{total}] HATA {name} × {gid}: {e}')

# Satır = kişi; ilk sütun girdi fotoğrafı, sonrakiler seçilen giysiler (sıra sabit)
TH_W = 240
for name, pp in persons.items():
    row = [('girdi', pp.image)] + [(g.split('/')[-1], results[(name, g)])
                                   for g in GARMENT_LIST if (name, g) in results]
    if len(row) < 2:
        continue
    display(Markdown(f"**{name}** — " + "  |  ".join(lbl for lbl, _ in row)))
    strip = Image.new('RGB', (TH_W * len(row), round(TH_W * size[0] / size[1])), 'white')
    for k, (_, arr) in enumerate(row):
        strip.paste(Image.fromarray(arr).resize((TH_W, strip.height), Image.LANCZOS), (k * TH_W, 0))
    display(strip)

print(f'\n{len(results)}/{total} üretildi → {OUT}')

In [ ]:
#@title 6) TEŞHİS — kayma gövdede mi, giyside mi?
# Dev gri leke / gövdeden taşan katman KOŞULLAMA hatasıdır, model hatası değil.
# Bu hücre iki arızayı AYIRIR:
#   (a) kamera bozuk  -> HMR2 gövdesi de kaymış (depth paneli kişiyle örtüşmüyor)
#   (b) giysi askısı  -> gövde doğru ama giysi yukarıda/aşağıda (hang_pad/binding)
# ÖNCE hücre 5'i çalıştırın (prep, hmr2, assets buradan gelir).

import numpy as np, cv2
from PIL import Image
from IPython.display import display, Markdown

DIAG_PERSON = PERSON_IMAGES[0]
DIAG_GARMENT = GARMENT_LIST[0]
ATR_GARMENT = (4, 7)  # parser'da üst giyim etiketleri

_pp = prep.process(DIAG_PERSON, size=size)
_params = hmr2(_pp.image, bbox=person_square_bbox(_pp))
_b = build_conditioning(_pp.image, _params, assets[DIAG_GARMENT], PhotoView(),
                        size=size, person_prep=_pp)

H, W = size
to_img = lambda t: ((t.numpy().transpose(1, 2, 0) + 1) / 2 * 255).clip(0, 255).astype(np.uint8)
depth = _b.control_depth_sil[0].numpy()
body = depth > -0.99                                    # HMR2 gövdesinin ekran izi
sil = _b.control_depth_sil[2].numpy() > 0               # mesh giysi silueti
mask = _b.inpaint_mask.numpy()[0] > 0.5
worn = np.isin(cv2.resize(np.asarray(_pp.parse), (W, H), interpolation=cv2.INTER_NEAREST),
               ATR_GARMENT)                             # kişinin GERÇEK giysisi (parser)

def overlay(base, m, color):
    o = base.copy()
    o[m] = (0.5 * o[m] + 0.5 * np.array(color)).astype(np.uint8)
    return o

panels = [("kişi", _pp.image),
          ("GÖVDE (HMR2)", overlay(_pp.image, body, (0, 160, 255))),
          ("GİYSİ MESH", overlay(_pp.image, sil, (255, 0, 0))),
          ("gerçek giysi (parser)", overlay(_pp.image, worn, (0, 255, 0))),
          ("maske", (mask * 255).astype(np.uint8)),
          ("agnostic", to_img(_b.agnostic_rgb))]
TH = 200
strip = Image.new('RGB', (TH * len(panels), round(TH * H / W)), 'white')
for k, (_, arr) in enumerate(panels):
    strip.paste(Image.fromarray(arr).convert('RGB').resize((TH, strip.height), Image.LANCZOS), (k * TH, 0))
display(Markdown("  |  ".join(l for l, _ in panels)))
display(strip)

cy = lambda m: (np.argwhere(m)[:, 0].mean() / H * 100) if m.any() else float('nan')
print(f"maske alanı  = %{mask.mean() * 100:.1f}   silüet alanı = %{sil.mean() * 100:.1f}")
print(f"dikey merkez (görüntü yüksekliğinin %'si, küçük = yukarıda):")
print(f"  gövde(HMR2)={cy(body):5.1f}   giysi-mesh={cy(sil):5.1f}   gerçek-giysi(parser)={cy(worn):5.1f}")
dy = cy(sil) - cy(worn)
print(f"  giysi-mesh ile gerçek-giysi arası dikey kayma = {dy:+.1f} puan")

if abs(dy) > 6:
    yon = "YUKARI" if dy < 0 else "AŞAĞI"
    print(f"\nTEŞHİS: giysi mesh'i gerçek giysiye göre {yon} kaymış ({abs(dy):.1f} puan).")
    print("  GÖVDE paneline bakın: mavi bölge kişiyle örtüşüyorsa kamera doğru, sorun")
    print("  giysi askısında (hang_pad/binding). Mavi de kaymışsa HMR2 kamerası bozulmuş —")
    print("  sebep genelde kırpma/oran; önden, tam gövde, 3:4 foto ile tekrar deneyin.")
else:
    print("\nHizalama makul; sorun üretim kalitesinde olabilir (GUIDANCE 3.5-7 deneyin).")